In [23]:
# Run this in Cell 1
%pip install librosa numpy pandas matplotlib seaborn tensorflow scikit-learn tqdm resampy

print("✅ Installation complete. Please RESTART your kernel now.")

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
✅ Installation complete. Please RESTART your kernel now.


In [1]:
import os
import pandas as pd
import librosa
import numpy as np
from tqdm import tqdm

print("--- DIAGNOSTIC CHECK ---")

# 1. Check Libraries
print(f"Librosa version: {librosa.__version__}")
print("✅ Libraries imported successfully.")

# 2. Check Paths
base_path = os.getcwd()
print(f"Current Directory: {base_path}")

audio_path = os.path.join(base_path, 'UrbanSound8K', 'audio')
metadata_path = os.path.join(base_path, 'UrbanSound8K', 'metadata', 'UrbanSound8K.csv')

if os.path.exists(audio_path) and os.path.exists(metadata_path):
    print("✅ Dataset found!")
    print(f"Audio folder: {audio_path}")
    
    # Check for Capitalization issues (fold1 vs Fold1)
    folders = os.listdir(audio_path)
    if 'fold1' in folders:
        print("✅ Sub-folders are lowercase ('fold1'). Perfect.")
    elif 'Fold1' in folders:
        print("⚠️ Sub-folders are Capitalized ('Fold1'). The code will need to adapt.")
    else:
        print("❌ Audio folder exists but seems empty or weird.")
else:
    print("❌ ERROR: Dataset NOT found. Check your folder structure.")

--- DIAGNOSTIC CHECK ---
Librosa version: 0.11.0
✅ Libraries imported successfully.
Current Directory: c:\Users\aryan\OneDrive\Desktop\python\Urban Sound project
✅ Dataset found!
Audio folder: c:\Users\aryan\OneDrive\Desktop\python\Urban Sound project\UrbanSound8K\audio
✅ Sub-folders are lowercase ('fold1'). Perfect.


In [2]:
# Define the Extractor Function
def features_extractor(file_name):
    try:
        # Load audio (kaiser_fast is faster for large datasets)
        audio, sample_rate = librosa.load(file_name, res_type='kaiser_fast') 
        
        # Extract features (MFCCs)
        mfccs_features = librosa.feature.mfcc(y=audio, sr=sample_rate, n_mfcc=40)
        
        # Scale (Average over time)
        mfccs_scaled_features = np.mean(mfccs_features.T, axis=0)
        return mfccs_scaled_features
    except Exception as e:
        # If a file is corrupt, just return None (don't crash)
        return None

# Load Metadata
metadata = pd.read_csv(metadata_path)
extracted_features = []

print("--- STARTING EXTRACTION LOOP ---")
# Use tqdm for a progress bar
for index_num, row in tqdm(metadata.iterrows()):
    
    # Construct the file path
    # NOTE: If Step 2 said your folders are 'Fold1', change 'fold' to 'Fold' below!
    fold_name = 'fold' + str(row["fold"]) 
    file_name = os.path.join(audio_path, fold_name, str(row["slice_file_name"]))
    
    final_class_labels = row["class"]
    
    # Extract
    data = features_extractor(file_name)
    
    # Store if successful
    if data is not None:
        extracted_features.append([data, final_class_labels])

print("✅ Extraction Finished!")

# Save to file
extracted_features_df = pd.DataFrame(extracted_features, columns=['feature', 'class'])
extracted_features_df.to_pickle("extracted_features.pkl")
print("✅ Data saved to 'extracted_features.pkl'")

--- STARTING EXTRACTION LOOP ---


0it [00:00, ?it/s]c:\ProgramData\anaconda3\Lib\site-packages\paramiko\pkey.py:82: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from this module in 48.0.0.
  "cipher": algorithms.TripleDES,
c:\ProgramData\anaconda3\Lib\site-packages\paramiko\transport.py:219: CryptographyDeprecationWarning: Blowfish has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.Blowfish and will be removed from this module in 45.0.0.
  "class": algorithms.Blowfish,
c:\ProgramData\anaconda3\Lib\site-packages\paramiko\transport.py:243: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from this module in 48.0.0.
  "class": algorithms.TripleDES,
3555it [03:39, 16.67it/s]C:\Users\aryan\AppData\Roaming\Python\Python312\site-packages\librosa\core\spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=13

✅ Extraction Finished!
✅ Data saved to 'extracted_features.pkl'


In [3]:
import pandas as pd
import numpy as np
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split

# 1. Load the data we just saved
extracted_features_df = pd.read_pickle("extracted_features.pkl")
print("Data loaded!")

# 2. Separate Features (X) and Labels (y)
X = np.array(extracted_features_df['feature'].tolist())
y = np.array(extracted_features_df['class'].tolist())

# 3. Encode the Labels
# The computer can't understand "dog_bark". It needs numbers (0, 1, 2...).
# We use LabelEncoder to turn "dog_bark" into numbers, 
# and to_categorical to turn those numbers into "One-Hot Encoding".
from sklearn.preprocessing import LabelEncoder
labelencoder = LabelEncoder()
y = to_categorical(labelencoder.fit_transform(y))

# 4. Split into Training (80%) and Testing (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

print(f"Training samples: {X_train.shape[0]}")
print(f"Testing samples: {X_test.shape[0]}")
print(f"Input shape for the model: {X_train.shape[1]}")

Data loaded!
Training samples: 6985
Testing samples: 1747
Input shape for the model: 40


In [4]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Activation, Flatten
from tensorflow.keras.optimizers import Adam
from sklearn import metrics

# 1. Define the number of classes (10 sounds)
num_labels = y.shape[1]

# 2. Build the Model
model = Sequential()

# First Layer (100 neurons)
model.add(Dense(100, input_shape=(40,)))
model.add(Activation('relu'))
model.add(Dropout(0.5))

# Second Layer (200 neurons)
model.add(Dense(200))
model.add(Activation('relu'))
model.add(Dropout(0.5))

# Third Layer (100 neurons)
model.add(Dense(100))
model.add(Activation('relu'))
model.add(Dropout(0.5))

# Output Layer (10 neurons)
model.add(Dense(num_labels))
model.add(Activation('softmax')) # Softmax gives us probabilities (e.g., 90% dog, 10% siren)

# 3. Compile the Model
model.compile(loss='categorical_crossentropy', metrics=['accuracy'], optimizer='adam')

print("Model Created!")
model.summary()

Model Created!


C:\Users\aryan\AppData\Roaming\Python\Python312\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 100)            │         4,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 100)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 100)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 200)            │        20,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 200)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 200)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 100)            │        20,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_2 (Activation)       │ (None, 100)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 100)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 10)             │         1,010 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_3 (Activation)       │ (None, 10)             │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 45,410 (177.38 KB)

 Trainable params: 45,410 (177.38 KB)

 Non-trainable params: 0 (0.00 B)

In [5]:
print("Starting Training...")

# Train for 100 epochs with a batch size of 32
num_epochs = 100
num_batch_size = 32

history = model.fit(X_train, y_train, batch_size=num_batch_size, epochs=num_epochs, validation_data=(X_test, y_test), verbose=1)

print("Training Finished!")

Starting Training...
Epoch 1/100
219/219 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.1173 - loss: 10.8678 - val_accuracy: 0.1070 - val_loss: 2.2955
Epoch 2/100
219/219 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.1237 - loss: 2.5407 - val_accuracy: 0.1048 - val_loss: 2.2833
Epoch 3/100
219/219 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.1225 - loss: 2.3299 - val_accuracy: 0.1059 - val_loss: 2.2752
Epoch 4/100
219/219 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.1248 - loss: 2.3007 - val_accuracy: 0.1093 - val_loss: 2.2668
Epoch 5/100
219/219 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.1284 - loss: 2.2767 - val_accuracy: 0.1534 - val_loss: 2.2321
Epoch 6/100
219/219 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.1424 - loss: 2.2454 - val_accuracy: 0.1809 - val_loss: 2.1899
Epoch 7/100
219/219 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.1512 - loss: 2.2180 - val_accuracy: 0.1935 - val_loss: 2.1632
Epoch 8/100
219/219 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.1666 - l

In [ ]:
# Save the trained model
model.save("urban_sound_classifier.h5")
print("✅ Model saved as 'urban_sound_classifier.h5'")

✅ Model saved as 'urban_sound_classifier.h5'


In [ ]:
import numpy as np
import librosa

# 1. Choose a file to test
# This file (101415-3-0-2.wav) is known to be a Dog Bark
filename = 'UrbanSound8K/audio/fold1/101415-3-0-2.wav' 

# 2. Define the Pre-processing function (Same as training)
def prepare_audio(file_path):
    # Load the audio
    audio, sample_rate = librosa.load(file_path, res_type='kaiser_fast') 
    # Extract features
    mfccs_features = librosa.feature.mfcc(y=audio, sr=sample_rate, n_mfcc=40)
    # Scale (mean)
    mfccs_scaled_features = np.mean(mfccs_features.T, axis=0)
    # Reshape to (1, 40) because the model expects a batch of data
    return mfccs_scaled_features.reshape(1, -1)

# 3. Predict
print(f"🎧 Listening to: {filename} ...")
prediction_feature = prepare_audio(filename)

# Get the probabilities
predictions = model.predict(prediction_feature) 
predicted_class_id = np.argmax(predictions, axis=1)[0] # Get the highest score

# 4. Decode the result
class_map = {
    0: "Air Conditioner",
    1: "Car Horn",
    2: "Children Playing",
    3: "Dog Bark",
    4: "Drilling",
    5: "Engine Idling",
    6: "Gun Shot",
    7: "Jackhammer",
    8: "Siren",
    9: "Street Music"
}

result = class_map[predicted_class_id]
confidence = predictions[0][predicted_class_id] * 100

print(f"\n🤖 AI Prediction: {result}")
print(f"📊 Confidence: {confidence:.2f}%")

if result == "Dog Bark":
    print("✅ SUCCESS! The model works.")
else:
    print("❌ INCORRECT. The AI got confused.")

🎧 Listening to: UrbanSound8K/audio/fold1/101415-3-0-2.wav ...
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 109ms/step

🤖 AI Prediction: Dog Bark
📊 Confidence: 100.00%
✅ SUCCESS! The model works.
